In [27]:
%pip install -q langchain-groq

In [4]:
from google.colab import userdata
from langchain_groq import ChatGroq

groq_llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=userdata.get("GEMINI_API_KEY"),
    temperature=0,
    max_retries=2,
)

In [5]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """You are a friendly assistant for Fresh Basket Vegetable Shop.

Prices per kilogram:
- Tomatoes: Rs. 40
- Onions: Rs. 35
- Potatoes: Rs. 30
- Carrots: Rs. 60
- Cucumbers: Rs. 45
- Green beans: Rs. 80

Customers can request in-store pickup. The shop confirms the final price,
stock, and pickup time.

Rules:
- Recommend vegetables based on the customer's recipe or budget.
- Ask for the quantity and unit when you need them for an estimate.
- Convert grams to kilograms before calculating. For example, 500 g is 0.5 kg.
- Use the listed prices for estimates and show the total in rupees.
- Refer unknown prices and stock details to the shop.
- Only confirm an order or pickup time after the shop verifies it.
- Answer questions about vegetables, prices, pickup, and storage.
- Remember the customer's name, budget, and order details from the conversation.
- Reply politely in English or Telugu, matching the customer.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

fresh_basket_chain = prompt | groq_llm | StrOutputParser()
history = []


def ask(question: str) -> str:
    answer = fresh_basket_chain.invoke({
        "history": history,
        "question": question,
    })
    history.extend([("human", question), ("ai", answer)])
    return answer

In [13]:
from google.colab import userdata
from groq import Groq, AuthenticationError

client = Groq(api_key=userdata.get("GROK_API_KEY").strip())

try:
    client.models.list()
except AuthenticationError:
    print("Groq rejected the key currently saved in Colab Secrets.")
else:
    print("Groq accepted the key.")

Groq accepted the key.


In [17]:
from google.colab import userdata
from groq import AuthenticationError
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

groq_llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=userdata.get("GROK_API_KEY").strip(),
    temperature=0,
)

fresh_basket_chain = prompt | groq_llm | StrOutputParser()

try:
    print(fresh_basket_chain.invoke({
        "history": [],
        "question": "How much does 1 kg of tomatoes cost?",
    }))
except AuthenticationError:
    print("Groq rejected the key currently saved as GROQ_API_KEY.")

Tomatoes cost Rs. 40 per kilogram.


In [18]:
import csv
import re
from decimal import Decimal

test_path = "/content/fresh_basket_tests.csv"

test_cases = [
    ("How much does 1 kg of tomatoes cost?", "40"),
    ("How much does 1 kg of onions cost?", "35"),
    ("How much does 1 kg of potatoes cost?", "30"),
    ("How much does 1 kg of carrots cost?", "60"),
    ("How much does 1 kg of cucumbers cost?", "45"),
    ("How much does 1 kg of green beans cost?", "80"),
    ("How much do 2 kg of tomatoes cost?", "80"),
    ("How much do 2 kg of onions cost?", "70"),
    ("How much do 3 kg of potatoes cost?", "90"),
    ("How much do 2 kg of cucumbers cost?", "90"),
    ("How much does 500 g of carrots cost?", "30"),
    ("How much does 250 g of green beans cost?", "20"),
    ("How much does 500 g of cucumbers cost?", "22.50"),
    ("How much does 750 g of carrots cost?", "45"),
    ("What is the total for 1 kg of tomatoes and 500 g of carrots?", "70"),
    ("What is the total for 2 kg of tomatoes and 1 kg of onions?", "115"),
    ("I have Rs. 100. What is the total for 1 kg each of tomatoes and potatoes?", "70"),
    ("అర కిలో క్యారెట్లు ఎంత? ధరను ₹ గుర్తుతో అంకెల్లో చెప్పండి.", "30"),
]

with open(test_path, "w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["question", "must_include"])
    writer.writerows(test_cases)


def run_tests(path=test_path):
    with open(path, encoding="utf-8-sig", newline="") as file:
        rows = list(csv.DictReader(file))

    if not rows:
        print("No test cases found in the CSV.")
        return

    passed = failed = errors = 0

    for row in rows:
        question = row["question"]
        expected = Decimal(row["must_include"])

        try:
            answer = fresh_basket_chain.invoke({
                "history": [],
                "question": question,
            })
        except Exception as exc:
            errors += 1
            print(f"ERROR | {question} → {exc}")
            continue

        # Check currency amounts so weights such as 500 g cannot pass a price test.
        amounts = re.findall(
            r"(?:₹|Rs\.?|INR)\s*(\d[\d,]*(?:\.\d+)?)"
            r"|(\d[\d,]*(?:\.\d+)?)\s*(?:rupees|రూపాయలు)",
            answer.replace("*", ""),
            flags=re.IGNORECASE,
        )
        ok = any(
            Decimal((before or after).replace(",", "")) == expected
            for before, after in amounts
        )

        passed += int(ok)
        failed += int(not ok)
        status = "PASS" if ok else "FAIL"
        preview = " ".join(answer.split())[:100]
        print(f"{status} | {question} → {preview}")

    print(f"\nPassed: {passed}")
    print(f"Failed: {failed}")
    print(f"Errors: {errors}")
    print(f"Pass Rate: {passed / len(rows):.0%}")


run_tests()

PASS | How much does 1 kg of tomatoes cost? → Tomatoes cost Rs. 40 per kilogram.
PASS | How much does 1 kg of onions cost? → Onions cost Rs. 35 per kilogram.
PASS | How much does 1 kg of potatoes cost? → 1 kg of potatoes costs **Rs. 30**.
PASS | How much does 1 kg of carrots cost? → Carrots cost **Rs. 60 per kilogram**.
PASS | How much does 1 kg of cucumbers cost? → Cucumbers cost **Rs. 45 per kilogram**.
PASS | How much does 1 kg of green beans cost? → Green beans cost **Rs. 80 per kilogram**.
PASS | How much do 2 kg of tomatoes cost? → 2 kg of tomatoes will cost you Rs. 80.
PASS | How much do 2 kg of onions cost? → Onions cost Rs. 35 per kilogram, so 2 kg would be Rs. 70.
PASS | How much do 3 kg of potatoes cost? → 3 kg of potatoes will cost Rs. 90.
PASS | How much do 2 kg of cucumbers cost? → 2 kg of cucumbers will cost Rs. 90.
PASS | How much does 500 g of carrots cost? → 500 g of carrots cost Rs. 30.
PASS | How much does 250 g of green beans cost? → Green beans cost Rs. 80 per kil

In [20]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=userdata.get("GEMINI_API_KEY"),
    temperature=0.3,
)

In [23]:
from langchain_core.output_parsers import StrOutputParser

fresh_basket_chain = prompt | llm | StrOutputParser()

print(fresh_basket_chain.invoke({
    "history": [],
    "question": "How much does 1 kg of tomatoes cost?",
}))

Hello! Thank you for reaching out to Fresh Basket Vegetable Shop.

1 kg of tomatoes costs **Rs. 40**. 

Would you like to add some tomatoes to your order, or is there anything else I can help you with today?


In [25]:
import csv
import time

def run_tests(test_chain, path="/content/test.csv"):
    try:
        with open(path, encoding="utf-8-sig", newline="") as file:
            rows = list(csv.DictReader(file))
    except FileNotFoundError:
        print(f"CSV not found: {path}. Run your CSV creation cell first.")
        return

    if not rows:
        print("No test questions found in the CSV.")
        return

    passed = failed = errors = 0

    for row in rows:
        question = row["question"]
        expected = row["must_include"].strip().lower().replace(",", "")

        for attempt in range(3):
            try:
                answer = test_chain.invoke({
                    "history": [],
                    "question": question,
                })
            except Exception as exc:
                if "503" in str(exc) and attempt < 2:
                    print("Server busy. Retrying in 15 seconds...")
                    time.sleep(15)
                    continue

                errors += 1
                print(f"ERROR | {question} → {exc}")
                break

            ok = expected in answer.lower().replace(",", "")
            passed += int(ok)
            failed += int(not ok)
            status = "PASS" if ok else "FAIL"
            preview = " ".join(answer.split())[:100]
            print(f"{status} | {question} → {preview}")
            break

    print(f"\nPassed: {passed}")
    print(f"Failed: {failed}")
    print(f"Errors: {errors}")
    print(f"Pass Rate: {passed / len(rows):.0%}")

run_tests(fresh_basket_chain)

Server busy. Retrying in 15 seconds...
PASS | What is the price of 1 kg of onions? → Hello! Welcome to Fresh Basket Vegetable Shop. The price for 1 kg of onions is **Rs. 35**. Would you
PASS | Can I collect my vegetables from the shop? → Yes, you certainly can! We offer in-store pickup for your convenience. Once you let me know what you
PASS | Can I pick up 2 kg of potatoes this evening? → Hello! I would be happy to help you with that. For 2 kg of potatoes, the price would be Rs. 60 (2 kg
PASS | How much does 250 g of green beans cost? → Hello! I'd be happy to help you with that. Since green beans are priced at Rs. 80 per kilogram, 250 
PASS | What is the total for 2 kg of tomatoes and 1 kg of onions? → Hello! I'd be happy to help you with that. Here is the price breakdown for your order: * **Tomatoes:

Passed: 5
Failed: 0
Errors: 0
Pass Rate: 100%


In [26]:
from langchain.agents import create_agent

agent = create_agent(model=llm, tools=[], system_prompt=SYSTEM)

result = agent.invoke({"messages": [{"role": "user", "content": "I have a budget of Rs 1000. Can I buy two Sunflower Bouquets? How much money will I have left?"}]})

print(result["messages"][-1].text)

Hello! Welcome to Fresh Basket Vegetable Shop. I would be happy to help you with your shopping today.

Regarding your request, we specialize in fresh vegetables, and unfortunately, we do not stock flowers like Sunflower Bouquets. Therefore, I cannot provide a price for them or confirm their availability.

However, if you would like to use your budget of Rs. 1000 to stock up on some fresh vegetables, I would be more than happy to help you with that! We have tomatoes, onions, potatoes, carrots, cucumbers, and green beans available.

Would you like to see a list of our vegetable prices or perhaps get some recommendations for a recipe?
